# Correction of the CaliberMRI phantom ADC maps

Applies the calibration to an independent target: the CaliberMRI phantom. For each diffusion-weighted scan the b-value map is
predicted from the duration-level calibration maps saved by `estimate_calibration.py` (`config.CALIB_DIR`), resampled to the
phantom grid, and used to calculate ADC maps in place of the nominal b-value. The traditional (nominal b-value) and calibrated
ADC maps are compared in the figure, and the distributions of the ADC in the background region of the phantom are compared.

The phantom segmentation uses the `ghost` library (not needed for the ADC maps and the figure).

In [ ]:
import config   # settings shared by all analysis scripts (also makes the root modules importable)

# Standard libraries
import numpy as np
from pathlib import Path
from glob import glob
import plotly.express as px
import matplotlib.pyplot as plt
from tqdm import tqdm
from functools import partial
import pandas as pd
import ants
import seaborn as sns

# Specific libraries
import nibabel as nib
from scipy.ndimage import distance_transform_edt
import colorcet

# custom functions
import registration as reg
import calib_utils as cal

# Ghost libraries
from ghost.phantom import Caliber137

In [ ]:
parent_dir = config.CALIBER_DIR
eval_dir = parent_dir / "derivatives"
plot_dir = config.PLOT_DIR
plot_dir.mkdir(exist_ok=True)
testname = eval_dir.parent.stem

eval_params_fname = eval_dir / (testname + "_" + "params.h5")
eval_scan_fname = eval_dir / (testname + "_" + "RAW_REG.nii.gz")
eval_mask_fname = eval_dir / (testname + "_" + "MASK.nii.gz")

# Load in sequence parameters
test_df = pd.read_hdf(eval_params_fname, key="df", mode="r")

# Load in scan data and add into data frame
test_nii = nib.load(eval_scan_fname)
test_data = test_nii.get_fdata()
test_aff = test_nii.affine
test_spacing = reg.spacing_from_affine(test_aff)

test_data_reshuffled = np.transpose(test_data,[3,0,1,2]).astype(np.float32)
test_df["image"] = list(test_data_reshuffled)

# Load in mask
eval_mask_nii = nib.load(eval_mask_fname)
eval_mask = eval_mask_nii.get_fdata().astype(bool)

In [ ]:
test_df.sort_values("scan_time", inplace=True, ignore_index=True)

# Put the b=0 image on b=0 rows, NaN elsewhere
b0_threshold = 5
test_df["s0_image"] = test_df["image"].where(test_df["b_value"] < b0_threshold)

# Forward-fill so each DWI scan gets the most recent preceding b=0 image
test_df["s0_image"] = test_df["s0_image"].ffill()

# Keep only diffusion-weighted scans
dwi_df = test_df[test_df["b_value"] > b0_threshold].copy()

# Optional safety check: drop rows that do not have a preceding b=0 scan
dwi_df = dwi_df[dwi_df["s0_image"].notna()].copy()

In [ ]:
# Prepare empty columns
dwi_df["mask"] = None
dwi_df["spacing"] = None
dwi_df["pred_bmap"] = None

for idx, row in dwi_df.iterrows():
    dur = row["dwi_grad_duration"]
    
    # Calibration maps of this gradient duration (from estimate_calibration.py)
    A_path = config.a_dur_fname(dur)
    
    # Unpack calibration data
    A, mask, sigma, spacing = cal.unpack_A_nii(A_path)
    
    # Compute pred_bmap
    pred_bmap = cal.est_bmap(
        row["grad_vec"],
        delta_small=dur,
        delta_big=row["TE"] / 2,
        A=A,
        mask=mask,
        sigma=sigma
    )
    
    # Store results
    dwi_df.at[idx, "mask"] = mask
    dwi_df.at[idx, "spacing"] = spacing
    dwi_df.at[idx, "pred_bmap"] = pred_bmap

In [ ]:
n_scans = len(dwi_df)

s_scans_list = []
s0_scans_list = []
pred_bmaps_resampled_list = []
masks_resampled_list = []

for i, row in dwi_df.iterrows():
    # Extract per-row data
    s = row["image"]
    s0 = row["s0_image"]
    pred_bmap = row["pred_bmap"]
    mask = row["mask"]
    spacing = row["spacing"]

    # Build resampler for THIS row
    resample_func = partial(
        reg.resample_array,
        current_spacing=spacing,
        target_spacing=test_spacing,
        interp_type=0
    )

    # Resample bmap and mask
    pred_bmap_resampled = resample_func(pred_bmap)
    mask_resampled = resample_func(mask)

    # Store
    s_scans_list.append(s)
    s0_scans_list.append(s0)
    pred_bmaps_resampled_list.append(pred_bmap_resampled)
    masks_resampled_list.append(mask_resampled)

# Stack original scans
s_scans = np.stack(s_scans_list, axis=-1)
s0_scans = np.stack(s0_scans_list, axis=-1)
pred_bmaps_resampled = np.stack(pred_bmaps_resampled_list, axis=-1)
masks_resampled = np.stack(masks_resampled_list, axis=-1)

b_values = dwi_df["b_value"].values
b_values_reshaped = b_values.reshape((1, 1, 1, -1))
dw_axes = dwi_df["dwi_direction"].values

# --- Crop to common overlap ---
overlap_shape = [
    min(a, b)
    for a, b in zip(pred_bmaps_resampled[..., -1].shape, s_scans[..., 0].shape)
]

s_scans_cropped = np.stack(
    [cal.center_crop_3d(s_scans[..., i], overlap_shape) for i in range(n_scans)],
    axis=-1
)

s0_scans_cropped = np.stack(
    [cal.center_crop_3d(s0_scans[..., i], overlap_shape) for i in range(n_scans)],
    axis=-1
)

pred_bmaps_cropped = np.stack(
    [cal.center_crop_3d(pred_bmaps_resampled[..., i], overlap_shape) for i in range(n_scans)],
    axis=-1
)

masks_cropped = np.stack(
    [cal.center_crop_3d(masks_resampled[..., i], overlap_shape) for i in range(n_scans)],
    axis=-1
)

# --- Build valid mask (now per scan) ---
valid_mask = (eval_mask[..., None] & masks_cropped.astype(bool))

# --- ADC calculations ---
adc_trad = np.zeros_like(s_scans_cropped)
adc_trad[valid_mask] = np.nan_to_num(
    np.log(s0_scans_cropped[valid_mask] / s_scans_cropped[valid_mask]) / b_values.repeat(np.prod(overlap_shape)).reshape(valid_mask.shape)[valid_mask],
    nan=0, posinf=0, neginf=0
)

adc_calib = np.zeros_like(s_scans_cropped)
adc_calib[valid_mask] = np.nan_to_num(
    np.log(s0_scans_cropped[valid_mask] / s_scans_cropped[valid_mask]) / pred_bmaps_cropped[valid_mask],
    nan=0, posinf=0, neginf=0
)

# --- Average repeats ---
adc_trad_avg = np.zeros([*adc_trad.shape[:-1], 3])
adc_calib_avg = np.zeros([*adc_trad.shape[:-1], 3])

for i, a in enumerate(np.unique(dw_axes)):
    rep_mask = dw_axes == a
    adc_trad_avg[..., i] = np.mean(adc_trad[..., rep_mask], axis=-1)
    adc_calib_avg[..., i] = np.mean(adc_calib[..., rep_mask], axis=-1)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(8, 8), facecolor="k")

colormap = colorcet.cm.rainbow_bgyr_10_90_c83
i_slice = 22
valid_mask_3d = np.all(valid_mask, axis=-1)

vmin=0.5
vmax=2.5
dif_lim = 0.4

fs_title = 15
fs_cbar = 10

labels = [rf"$ADC_{a}$" for a in np.unique(dw_axes)]

row_mappables = [None, None, None]
for i in range(3):
    trad_slice = adc_trad_avg[..., i_slice, i]
    calib_slice = adc_calib_avg[..., i_slice, i]
    trad_slice = np.rot90(trad_slice, k=1, axes=(0, 1))[7:-7, 6:-7]
    calib_slice = np.rot90(calib_slice, k=1, axes=(0, 1))[7:-7, 6:-7]

    trad_slice_m = trad_slice * 1e3
    calib_slice_m = calib_slice * 1e3

    mask_slice = np.rot90(valid_mask_3d[:, :, i_slice], k=1, axes=(0, 1))[7:-7, 6:-7]
    plot_mask = np.full_like(trad_slice, np.nan, dtype=float)
    plot_mask[mask_slice == 0] = 1

    im0 = axes[0, i].imshow(trad_slice_m, cmap=colormap, vmin=vmin, vmax=vmax)
    axes[0, i].imshow(plot_mask, cmap="Greys", vmin=0, vmax=1)
    im1 = axes[1, i].imshow(calib_slice_m, cmap=colormap, vmin=vmin, vmax=vmax)
    axes[1, i].imshow(plot_mask, cmap="Greys", vmin=0, vmax=1)
    im2 = axes[2, i].imshow(trad_slice_m - calib_slice_m, cmap=colorcet.cm.coolwarm, vmin=-dif_lim, vmax=dif_lim)
    axes[2, i].imshow(plot_mask, cmap="Greys", vmin=0, vmax=1)

    if i == 0:
        row_mappables[0] = im0
        row_mappables[1] = im1
        row_mappables[2] = im2
    axes[0, i].set_title(dw_axes[i], fontsize=fs_title, style="italic", color="white")

for ax in axes.flatten():
    ax.set_axis_off()

plt.tight_layout(rect=[0.15, 0, 0.95, 0.95])

# Add row labels on the left
row_labels = ["Uncalibrated", "Calibrated", "Difference"]
x_offset = -0.05

for r, label in enumerate(row_labels):
    # Get position of the leftmost subplot in this row
    pos = axes[r, 0].get_position()
    
    fig.text(
        pos.x0 - 0.015,                    # move slightly left of plots
        pos.y0 + pos.height / 2,           # vertically centered on row
        label,
        va="center",
        ha="right",
        fontsize=fs_title,
        rotation=90,
        style="italic",
        color="white"
    )

cbar_width = 0.01
cbar_pad = 0.05

for r in range(3):

    pos_left = axes[r, 0].get_position()
    pos_right = axes[r, -1].get_position()

    # vertical span of the row
    shrink = 0.7
    full_height = pos_left.height
    y0 = pos_left.y0 + full_height * (1-shrink)/ 2

    # position to the right of last column
    x0 = pos_right.x1 + cbar_pad

    cax = fig.add_axes([x0, y0, cbar_width, full_height * shrink])
    cbar = fig.colorbar(row_mappables[r], cax=cax)

    cbar.ax.tick_params(colors="white", labelsize=10)
    cbar.outline.set_edgecolor("white")

    if r == 2:
        cbar.ax.set_title("$\Delta\ ADC$ \n$( \\times 10^{-3}\ mm^2/s)$", color="white", pad=10, fontsize=fs_cbar, loc="center")
    else:
        cbar.ax.set_title("$ADC$ \n$( \\times 10^{-3}\ mm^2/s)$", color="white", pad=10, fontsize=fs_cbar, loc="center")
    
fmt = "png"
plot_fname = plot_dir / (testname + "_" + f"adc_map_comparison.{fmt}")
fig.savefig(fname=plot_fname, dpi=600, format=fmt, bbox_inches="tight", pad_inches=0.25)

In [ ]:
# Extract an avergae S0 image
s0_avg = test_df[test_df["b_value"]<b0_threshold]["image"].mean()
s0_a = ants.from_numpy(s0_avg, spacing=test_spacing)
# s0_a.plot_ortho(resample=True, flat=True)

# Register segmentations of different subvolumes to S0 image
phantom = Caliber137()
inv, fwd = phantom.reg_to_phantom(s0_a, do_syn=True, weighting='T2w',)

# Extract a register mask of the different sub-volumes in the Caliber Phantom
masks = np.zeros([*s0_avg.shape, 3], dtype=np.float32)
for i, weight in enumerate(["ADC", "T1mimics", "T2mimics"]):
    seg_a = phantom.warp_seg(s0_a, seg=weight, xfm=inv)
    masks[..., i] = seg_a.numpy()

    # Can plot each segmentation for confirmation of alignment
    # s0_a.plot(axis=2, overlay=seg_a, slices=np.arange(15, 35))

In [ ]:
# Combine sub-volume masks into single 'non-background' mask
combined_masks = np.any(masks, axis=3)

# Extract mask for only the flood filled regions of the phantom
background_mask = (valid_mask_3d) & (~combined_masks)

# Plot for confirmation
plot_data = np.concatenate([combined_masks, valid_mask_3d, background_mask, adc_trad_avg[..., 0]*0.4e3], axis=1)
px.imshow(plot_data, animation_frame=2, height=800)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(16,8))

for i, dataset in enumerate([adc_trad_avg, adc_calib_avg]):

    for j in range(3):
        
        bg_vals = dataset[..., j][background_mask] * 1e3

        bg_median = np.median(bg_vals)
        lq = np.percentile(bg_vals, 25)
        uq = np.percentile(bg_vals, 75)
        sigma = np.std(bg_vals)

        label = ["Trad", "Calib"][i] + "_" + "XYZ"[j]
        sns.kdeplot(bg_vals, clip=(1,3), color=[0, 0.2+(1-i)*j/3, 0.2+i*j/3], label=label)
        print(label)
        print(f"Median: {bg_median:.4f}, LQ: {lq:.4f}, UQ:{uq:.4f}, IQR:{(uq-lq):.4f}, std: {sigma:.4f}\n")

ax.legend()
ax.set_xlabel("ADC")
fig.show()


In [ ]:
adc_mask = phantom.warp_seg(s0_a, seg="ADC", xfm=inv)

In [ ]:
plotdata = np.concatenate([adc_mask.numpy()[..., 21], adc_calib_avg[...,21, 0]*4e3], axis=1)
plt.imshow(plotdata, vmin=0, vmax=15)

## Vial-wise comparison of the ADC with and without calibration

The CaliberMRI ADC vials contain six solutions (water and 10-50 % PVP), most in more than one vial at different positions
in the field of view (`config.CALIBER_ADC_LABELS`). If the calibration removes the position and direction dependence of the b-value, the
ADC measured in vials of the same solution should agree better across the field of view and across the three diffusion
directions. Each vial is eroded before the voxels are extracted to avoid boundary (partial volume / registration) voxels,
and the same voxels are used for the uncalibrated and calibrated ADC.

In [ ]:
from scipy.ndimage import binary_erosion, generate_binary_structure

VIAL_ERODE_VOXELS = 1   # erode each vial by this many voxels to avoid boundary voxels
MIN_VOXELS = 5          # vials with fewer voxels (after erosion and masking) are left out

axis_names = list(np.unique(dw_axes))     # order of the last axis of adc_trad_avg / adc_calib_avg
conditions = {"trad": "Uncalibrated", "calib": "Calibrated"}
adc_maps = {"trad": adc_trad_avg * 1e3, "calib": adc_calib_avg * 1e3}   # x10^-3 mm^2/s

# ADC segmentation warped onto the scan grid
adc_labels = np.rint(masks[..., 0]).astype(int)
if adc_labels.shape != valid_mask_3d.shape:
    raise ValueError(f"Segmentation {adc_labels.shape} and ADC maps {valid_mask_3d.shape} are on different grids")

expected = {l for labs in config.CALIBER_ADC_LABELS.values() for l in labs}
not_found = expected - set(np.unique(adc_labels))
if not_found:
    print(f"WARNING: labels not found in the warped segmentation: {sorted(not_found)}")

# Voxels with a usable (positive) ADC in every direction, calibrated and uncalibrated
usable = valid_mask_3d & np.all(adc_maps["trad"] > 0, axis=-1) & np.all(adc_maps["calib"] > 0, axis=-1)
structure = generate_binary_structure(3, 1)

vial_vox = {}      # (solution, label) -> {condition: [n_voxels, n_directions] array of ADC values}
count_rows = []
for solution, labs in config.CALIBER_ADC_LABELS.items():
    for lab in labs:
        vial = adc_labels == lab
        eroded = binary_erosion(vial, structure=structure, iterations=VIAL_ERODE_VOXELS) if VIAL_ERODE_VOXELS > 0 else vial
        sel = eroded & usable
        count_rows.append({"solution": solution, "label": lab, "voxels": int(vial.sum()),
                           "after erosion": int(eroded.sum()), "used": int(sel.sum()),
                           "included": bool(sel.sum() >= MIN_VOXELS)})
        if sel.sum() >= MIN_VOXELS:
            vial_vox[(solution, lab)] = {c: adc_maps[c][sel] for c in conditions}

# Background fluid (the region of the phantom outside all of the segmented sub-volumes), treated as one extra "vial"
bg = binary_erosion(background_mask, structure=structure, iterations=VIAL_ERODE_VOXELS) if VIAL_ERODE_VOXELS > 0 else background_mask
bg_sel = bg & usable
count_rows.append({"solution": "Background", "label": "background", "voxels": int(background_mask.sum()),
                   "after erosion": int(bg.sum()), "used": int(bg_sel.sum()), "included": bool(bg_sel.sum() >= MIN_VOXELS)})
if bg_sel.sum() >= MIN_VOXELS:
    vial_vox[("Background", "background")] = {c: adc_maps[c][bg_sel] for c in conditions}

counts_df = pd.DataFrame(count_rows)
print(counts_df.to_string(index=False))

solutions = [s for s in ["Background", *config.CALIBER_ADC_LABELS] if any(k[0] == s for k in vial_vox)]   # background first

In [ ]:
# Median ADC of every vial, direction and condition
rows = []
for (solution, lab), vox in vial_vox.items():
    for c, name in conditions.items():
        for d, axis in enumerate(axis_names):
            rows.append({"solution": solution, "vial": lab, "condition": name, "direction": axis,
                         "median": np.median(vox[c][:, d]), "n_voxels": vox[c].shape[0]})
vial_df = pd.DataFrame(rows)

# Consistency metrics (coefficients of variation, %) of the vial medians
metric_rows = []
for solution in solutions:
    for name in conditions.values():
        sub = vial_df[(vial_df["solution"] == solution) & (vial_df["condition"] == name)]
        table = sub.pivot(index="vial", columns="direction", values="median")    # vials x directions
        cv = lambda x, axis: 100 * np.std(x, axis=axis, ddof=1) / np.mean(x, axis=axis)
        metric_rows.append({"solution": solution, "condition": name, "n_vials": table.shape[0],
                            "mean ADC": table.values.mean(),
                            "CV between vials (%)": cv(table.values, 0).mean() if table.shape[0] > 1 else np.nan,
                            "CV between directions (%)": cv(table.values, 1).mean(),
                            "CV overall (%)": cv(table.values.ravel(), 0) if table.size > 1 else np.nan})
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(3))

metrics_fname = plot_dir / (testname + "_" + "vial_adc_metrics.csv")
metrics_df.to_csv(metrics_fname, index=False)

In [ ]:
# Figure 1: the voxel ADC inside the vials of each solution, per direction, uncalibrated vs calibrated.
# Boxes: all voxels of the vials of that solution. Markers: median of each vial (the background fluid is a single region).
# One colour per solution (water, the 0 % PVP, is the first colour of the map) for the calibrated data and the
# vial layout. The uncalibrated data are grey. The background region is probably water, so it uses the water colour.
from matplotlib.colors import to_rgba
from matplotlib.legend_handler import HandlerTuple
sol_colors = {s: to_rgba(c) for s, c in zip(config.CALIBER_ADC_LABELS, plt.cm.viridis(np.linspace(0.3, 0.95, len(config.CALIBER_ADC_LABELS))))}
box_colors = {**sol_colors, "Background": sol_colors[next(iter(sol_colors))]}
uncal_color = "0.7"
fill_alpha = 0.8     # opacity of the box fill (box outlines, medians and whiskers are black)
cond_color = lambda c, solution: uncal_color if c == "trad" else box_colors[solution]
vial_markers = ["o", "s", "^", "D"]
offset = 0.2

fig, axs = plt.subplots(2, 4, figsize=(16, 12))
for ax, solution in zip(axs.flat, solutions):
    vials = [lab for (s, lab) in vial_vox if s == solution]

    for ci, (c, name) in enumerate(conditions.items()):
        for d in range(len(axis_names)):
            pos = d + (ci - 0.5) * 2 * offset
            col = cond_color(c, solution)
            pooled = np.concatenate([vial_vox[(solution, lab)][c][:, d] for lab in vials])
            ax.boxplot(pooled, positions=[pos], widths=0.3, showfliers=False, patch_artist=True,
                       boxprops=dict(facecolor=to_rgba(col, fill_alpha), edgecolor="k"),
                       medianprops=dict(color="k", linewidth=1.5),
                       whiskerprops=dict(color="k"), capprops=dict(color="k"))
            for vi, lab in enumerate(vials):
                x = pos + (vi - (len(vials) - 1) / 2) * 0.06
                ax.plot(x, np.median(vial_vox[(solution, lab)][c][:, d]), marker=vial_markers[vi % 4],
                        color=col, markeredgecolor="k", markersize=6, linestyle="none", zorder=3)

    ax.set_xticks(range(len(axis_names)), axis_names)
    ax.set_xlim(-0.6, len(axis_names) - 0.4)
    ax.set_ylim(0, 3)
    ax.set_title(solution)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(handles=[plt.Line2D([], [], marker=vial_markers[vi % 4], color="0.5", markeredgecolor="k", linestyle="none",
                                  label=(lab if isinstance(lab, str) else f"Vial {lab}")) for vi, lab in enumerate(vials)],
              fontsize=9, loc="lower left", frameon=False)
for ax in axs[:, 0]:
    ax.set_ylabel(r"ADC ($\times 10^{-3}$ mm$^2$/s)", fontsize=13)
for ax in axs.flat[len(solutions) + 1:]:
    ax.set_axis_off()

# Layout of the vials in the next panel: the axial slice containing most of the vials, coloured by solution
# and annotated with the segmentation label of each vial (greyed out if the vial was left out of the analysis)
import matplotlib.patheffects as pe
from scipy.ndimage import center_of_mass

label_to_solution = {lab: s for s, labs in config.CALIBER_ADC_LABELS.items() for lab in labs}

n_in_slice = [len(set(np.unique(adc_labels[:, :, z])) & set(label_to_solution)) for z in range(adc_labels.shape[2])]
i_slice = int(np.argmax(n_in_slice))
lab_slice = np.rot90(adc_labels[:, :, i_slice])      # same orientation as the ADC map figure
absent = sorted(set(label_to_solution) - set(np.unique(lab_slice)))
if absent:
    print(f"Vials not in slice {i_slice} (not shown in the layout): {absent}")

ax = axs.flat[len(solutions)]
under = np.rot90(s0_avg[:, :, i_slice])
aspect = test_spacing[1] / test_spacing[0]    # rows are the second axis of the volume, columns the first
ax.imshow(under, cmap="gray", vmin=np.percentile(s0_avg, 1), vmax=np.percentile(s0_avg, 99), aspect=aspect)
overlay = np.zeros((*lab_slice.shape, 4))
for lab, s in label_to_solution.items():
    overlay[lab_slice == lab] = sol_colors[s]
ax.imshow(overlay, alpha=0.8, aspect=aspect)

included = {lab for (_, lab) in vial_vox}
for lab in label_to_solution:
    if lab in absent:
        continue
    i, j = center_of_mass(lab_slice == lab)
    ax.text(j, i, str(lab), ha="center", va="center", fontsize=10, fontweight="bold",
            color="white" if lab in included else "0.6", path_effects=[pe.withStroke(linewidth=2, foreground="k")])

# Show the whole phantom: a square field of view of LAYOUT_FOV_MM centred on the vials
LAYOUT_FOV_MM = 160
rows_, cols_ = np.nonzero(np.isin(lab_slice, list(label_to_solution)))
ci, cj = rows_.mean(), cols_.mean()
half_i, half_j = LAYOUT_FOV_MM / 2 / test_spacing[1], LAYOUT_FOV_MM / 2 / test_spacing[0]
ax.set_xlim(cj - half_j, cj + half_j)
ax.set_ylim(ci + half_i, ci - half_i)
ax.set_axis_off()
ax.set_title(f"Vial layout")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, facecolor=col, label=s) for s, col in sol_colors.items()],
          loc="upper center", bbox_to_anchor=(0.5, 0), ncol=2, fontsize=9, frameon=False)

# Legend of the conditions: grey = uncalibrated, solution colours = calibrated
patch = lambda col: plt.Rectangle((0, 0), 1, 1, facecolor=to_rgba(col, fill_alpha), edgecolor="k")
sample_cols = [list(sol_colors.values())[i] for i in (0, len(sol_colors) // 2, -1)]
fig.legend(handles=[patch(uncal_color), tuple(patch(col) for col in sample_cols)],
           labels=["Uncalibrated", "Calibrated (solution colour)"],
           handler_map={tuple: HandlerTuple(ndivide=None, pad=0)},
           loc="upper center", ncol=2, fontsize=13, frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.subplots_adjust(hspace=0.3)   # separate the two rows of panels

fig.savefig(plot_dir / (testname + "_" + "vial_adc_comparison.png"), dpi=300, bbox_inches="tight")